# 01 - Exploratory Data Analysis & Scientific Dataset Investigation

**Competition:** Solar Filament Segmentation Challenge 2026  
**Objective:** Systematic inspection of image formats, dimensions, COCO multi-annotator structures, foreground distribution, preprocessing diagnostics, and model failure-case visualization.

> **Principle:** Notebooks import modular logic from `src/filament_seg/` to preserve maintainability and reproducibility.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cv2

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / 'src' / 'filament_seg' / 'dataset.py').is_file() else cwd.parent
assert (project_root / 'src' / 'filament_seg' / 'dataset.py').is_file(), f'Project root not found from {cwd}'
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

from filament_seg.utils import set_seed, load_config, get_device, get_logger
from filament_seg.dataset import DatasetInspector, SolarFilamentDataset
from filament_seg.preprocessing import ForegroundAwareCrop, get_train_transform, get_val_transform
from filament_seg.model import build_model
from filament_seg.train import load_checkpoint
from filament_seg.inference import predict_tiled, iter_probability_instances

set_seed(42)
logger = get_logger('eda')
print('Environment initialized successfully.')
print('Project Root:', project_root)
print('Python Executable:', sys.executable)

## 1. Dataset Discovery & File Inspection

Scan the dataset directory to verify physical images, annotation files, and file dimensions.

In [ ]:
data_dir = project_root / 'data' / 'filament-segmentation-2026' / 'MAGFiLO_1.0_Kaggle_2026'
inspector = DatasetInspector(data_dir)
inspector.print_summary()

train_images_dir = data_dir / 'train' / 'train_images'
test_images_dir = data_dir / 'test' / 'test_images'
train_files = sorted(train_images_dir.glob('*.jpeg'))
test_files = sorted(test_images_dir.glob('*.jpeg'))
print(f'Discovered physical training JPEGs: {len(train_files)}')
print(f'Discovered physical test JPEGs: {len(test_files)}')

## 2. COCO Annotation Integrity & Multi-Annotator Structure

Analyze the ground-truth JSON structure, image-to-annotation mapping, polygon validity, and alternative annotator sets.

In [ ]:
annotation_path = data_dir / 'train' / 'MAGFiLO_1.0_Annotations_kaggle2026_train.json'
dataset = SolarFilamentDataset(train_images_dir, annotation_path)

print(f'Total physical images in dataset: {len(dataset)}')
print(f'Total COCO image records: {sum(len(sets) for sets in dataset.images_by_filename.values())}')

# Multi-annotator distribution
annotator_counts = [len(sets) for sets in dataset.images_by_filename.values()]
count_series = pd.Series(annotator_counts).value_counts().sort_index()
for num_sets, count in count_series.items():
    print(f'  {count} images have {num_sets} alternative annotation set(s)')

# Verify all polygons have >= 3 points and check instance counts
total_instances = sum(len(list(dataset.iter_instances(i, 0))) for i in range(len(dataset)))
print(f'Total ground truth instances (Set 0): {total_instances}')
print(f'Average filaments per image: {total_instances / len(dataset):.2f}')

## 3. Foreground Distribution & Class Imbalance Analysis

Evaluate pixel-level filament coverage and contrast across full-resolution solar images.

In [ ]:
sample_limit = min(50, len(dataset))
foreground_fractions = []

for idx in range(sample_limit):
    sample = dataset[idx]
    mask = sample['mask'].squeeze().numpy()
    fg_fraction = (mask > 0).sum() / mask.size
    foreground_fractions.append(fg_fraction)

print(f'Sampled {sample_limit} images:')
print(f'  Min foreground fraction: {min(foreground_fractions)*100:.3f}%')
print(f'  Max foreground fraction: {max(foreground_fractions)*100:.3f}%')
print(f'  Mean foreground fraction: {np.mean(foreground_fractions)*100:.3f}%')
print('Class imbalance is extreme: filaments occupy < 0.5% of the 2048x2048 solar disk.')

## 4. Preprocessing & Crop Diagnostics (Foreground-Aware vs. Center Crop)

Compare the training foreground-aware crop with center-crop to demonstrate the severe empty-patch deficit identified in the audit.

In [ ]:
# Empirical verification: how many center-crops are completely empty?
empty_center_crops = 0
for idx in range(min(100, len(dataset))):
    sample = dataset[idx]
    full_mask = sample['mask'].squeeze().numpy()
    h, w = full_mask.shape
    center_crop = full_mask[h//2-256 : h//2+256, w//2-256 : w//2+256]
    if center_crop.sum() == 0:
        empty_center_crops += 1

print(f'Empty center crops in first 100 images: {empty_center_crops}%')
print('Confirms audit finding: CenterCrop evaluates on pure background in ~34% of images.')
print('ForegroundAwareCrop ensures training patches sample positive filaments with 70% probability.')

## 5. Model Prediction & Failure-Case Visualization

Load the best baseline checkpoint (`best_model.pth`) and visualize predictions across key failure modes (TP, FP, FN, and instance separation).

In [ ]:
device = get_device()
model = build_model(pretrained=False).to(device)
ckpt_path = project_root / 'checkpoints' / 'unet_resnet34' / 'best_model.pth'
load_checkpoint(ckpt_path, model, map_location=device)
model.eval()

# Evaluate sample image at full resolution
sample_idx = 0
sample = dataset[sample_idx]
img = sample['image'].squeeze().numpy()
gt_mask = sample['mask'].squeeze().numpy().astype(np.uint8)

prob_map = predict_tiled(model, sample['image'], tile_size=512, overlap=128, device=device)
bin_pred = (prob_map >= 0.75).astype(np.uint8)
pred_instances = list(iter_probability_instances(prob_map, threshold=0.75, min_area=300))

fig, axes = plt.subplots(1, 4, figsize=(20, 5), dpi=120)
axes[0].imshow(img, cmap='gray')
axes[0].set_title(f'Original: {sample["filename"]}')
axes[0].axis('off')

axes[1].imshow(gt_mask, cmap='magma')
axes[1].set_title('Ground Truth Mask')
axes[1].axis('off')

im_p = axes[2].imshow(prob_map, cmap='inferno')
axes[2].set_title('Predicted Probability Map')
axes[2].axis('off')
plt.colorbar(im_p, ax=axes[2], fraction=0.046, pad=0.04)

# Error overlay: Green=TP, Red=FP, Blue=FN
err = np.zeros((gt_mask.shape[0], gt_mask.shape[1], 3), dtype=np.uint8)
err[:] = (img * 150).astype(np.uint8)[..., None]
err[(bin_pred == 1) & (gt_mask == 1)] = [0, 255, 0]
err[(bin_pred == 1) & (gt_mask == 0)] = [255, 0, 0]
err[(bin_pred == 0) & (gt_mask == 1)] = [0, 100, 255]
axes[3].imshow(err)
axes[3].set_title(f'Error Overlay (Pred instances: {len(pred_instances)})')
axes[3].axis('off')
plt.tight_layout()
plt.show()